<a href="https://colab.research.google.com/github/RafaelaMlucca/conformal-violence-prediction/blob/main/03_locart_aplicacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 03 — Locart adaptado para classificação binária

Implemento Locart (Cabezas et al., 2024) adaptado
para classificação binária seguindo a abordagem de Frohlich et al. (2025) no
PersonalizedUS, e aplico aos três desfechos de violência.


## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import pickle
import math
from collections import defaultdict
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

DRIVE = Path('/content/drive/MyDrive/projeto_violencia_mulher')
SAIDA = DRIVE / 'conformal_violence'

RANDOM_STATE = 42
ALPHA = 0.1   # cobertura nominal de 90%

## 2. Carregando dados e modelos

Reaproveito os modelos treinados no notebook 02 para não retreinar.

In [3]:
# dados
with open(SAIDA / 'dados_conformal.pkl', 'rb') as f:
    dados = pickle.load(f)

X_train = dados['X_train']
X_cal = dados['X_cal']
X_test = dados['X_test']
y_train = dados['y_train']
y_cal = dados['y_cal']
y_test = dados['y_test']
feature_names = dados['feature_names']
ALVOS = dados['alvos']

# modelos do notebook 02
with open(SAIDA / 'resultados_baseline.pkl', 'rb') as f:
    baseline = pickle.load(f)

modelos = baseline['modelos']
print('Modelos carregados:', list(modelos.keys()))

Modelos carregados: ['y_fisic', 'y_psico', 'y_sexu']


## 3. Função: aprende as folhas e quantis para um desfecho

Essa é a função central. Segue o esquema de (Frohlich et al. 2025):

- Calcula resíduos no conjunto de calibração
- Treina árvore de regressão sobre esses resíduos
- Mapeia cada amostra da calibração para sua folha
- Calcula quantil ajustado de cada folha


In [4]:
def aprender_locart(modelo, X_cal, y_cal_alvo, alpha=ALPHA,
                    max_depth=5, min_samples_leaf=None, random_state=RANDOM_STATE):

    # passo 1: resíduos
    proba_cal = modelo.predict_proba(X_cal)
    residuos = 1 - proba_cal[np.arange(len(y_cal_alvo)), y_cal_alvo]

    # passo 2: árvore sobre os resíduos
    # se min_samples_leaf não foi especificado, usa ~2% da calibração
    # (no Alek, com 1059 cal, ele usou 70-100, ou seja, 7-9%)
    if min_samples_leaf is None:
        min_samples_leaf = max(50, int(0.005 * len(X_cal)))

    arvore = DecisionTreeRegressor(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=random_state,
    )
    arvore.fit(X_cal, residuos)

    # passo 3: mapa de calibração -> folha
    folhas_cal = arvore.apply(X_cal)

    # passo 4: quantil por folha
    quantis = {}
    for folha in np.unique(folhas_cal):
        mascara = folhas_cal == folha
        probs_folha = residuos[mascara]
        n = len(probs_folha)
        if n == 0:
            continue
        cobertura_ajustada = math.ceil((n + 1) * (1 - alpha)) / n
        cobertura_ajustada = min(cobertura_ajustada, 1.0)
        # quantil dos resíduos -> threshold final
        # (1 - quantile) é o que comparamos com p(classe) na predição
        q_residuo = np.quantile(probs_folha, cobertura_ajustada, method='higher')
        quantis[int(folha)] = 1.0 - q_residuo

    return {
        'arvore': arvore,
        'quantis': quantis,
        'min_samples_leaf': min_samples_leaf,
        'n_folhas': arvore.get_n_leaves(),
    }

## 4. Função: gera conjuntos de predição usando Locart

Para cada caso de teste, descobre em qual folha cai, pega o quantil dessa folha,
e inclui no conjunto cada classe cuja probabilidade prevista é maior ou igual
ao quantil.

In [5]:
def predizer_locart(modelo, locart, X_test):
    """Gera conjuntos de predição usando Locart.

    Retorna:
        - conjuntos: array (n_test, 2) booleano
        - folhas_test: array (n_test,) com a folha de cada caso
    """
    arvore = locart['arvore']
    quantis = locart['quantis']

    proba_test = modelo.predict_proba(X_test)
    folhas_test = arvore.apply(X_test)

    n_test = len(X_test)
    conjuntos = np.zeros((n_test, 2), dtype=bool)

    for i in range(n_test):
        folha = int(folhas_test[i])
        q = quantis.get(folha, 0.5)  # fallback razoável se folha desconhecida
        for classe in [0, 1]:
            if proba_test[i, classe] >= q:
                conjuntos[i, classe] = True

    return conjuntos, folhas_test

## 5. Função auxiliar: avalia conjuntos

Mesma função do notebook 02. Coloco aqui para o notebook 03 ser independente.

In [6]:
def avaliar_conjuntos(y_verdadeiro, conjuntos):
    n = len(y_verdadeiro)
    cobertos = np.array([conjuntos[i, y_verdadeiro[i]] for i in range(n)])
    tamanhos = conjuntos.sum(axis=1)
    return {
        'cobertura': float(cobertos.mean()),
        'tamanho_medio': float(tamanhos.mean()),
        'pct_vazios': float((tamanhos == 0).mean()),
        'pct_unitarios': float((tamanhos == 1).mean()),
        'pct_duplos': float((tamanhos == 2).mean()),
    }

## 6. Rodando Locart para os 3 desfechos

Para cada desfecho: aprende as folhas e quantis na calibração, depois aplica no
teste. Imprime o número de folhas e a cobertura global.

In [7]:
resultados_locart = {}

for alvo in ALVOS:
    modelo = modelos[alvo]
    y_cal_alvo = y_cal[alvo].values
    y_test_alvo = y_test[alvo].values

    # aprende
    locart = aprender_locart(modelo, X_cal, y_cal_alvo)

    # prediz
    conjuntos, folhas_test = predizer_locart(modelo, locart, X_test)

    # avalia
    metricas = avaliar_conjuntos(y_test_alvo, conjuntos)

    resultados_locart[alvo] = {
        **metricas,
        'conjuntos': conjuntos,
        'folhas_test': folhas_test,
        'locart': locart,
    }

    print(f"{alvo}: {locart['n_folhas']} folhas | "
          f"cob={metricas['cobertura']:.4f} | "
          f"tam={metricas['tamanho_medio']:.4f} | "
          f"vazios={metricas['pct_vazios']:.1%}")

y_fisic: 28 folhas | cob=0.9034 | tam=1.1871 | vazios=2.2%
y_psico: 28 folhas | cob=0.9021 | tam=1.2349 | vazios=3.2%
y_sexu: 29 folhas | cob=0.8924 | tam=1.0135 | vazios=6.1%


## 7. Cobertura local: a contribuição real

A cobertura global por si só não diz muito (ela é parecida com MAPIE). O que
torna Locart útil é a **cobertura por subgrupo**. Aqui calculo a cobertura
empírica dentro de cada folha — e mostro também o tamanho do subgrupo e o
tamanho médio dos conjuntos lá dentro.

In [8]:
def cobertura_por_folha(y_test_alvo, conjuntos, folhas_test):
    """Calcula métricas dentro de cada folha."""
    folhas_unicas = np.unique(folhas_test)
    linhas = []
    for f in folhas_unicas:
        mask = folhas_test == f
        if mask.sum() == 0:
            continue
        y_f = y_test_alvo[mask]
        c_f = conjuntos[mask]
        cobertos = np.array([c_f[i, y_f[i]] for i in range(len(y_f))])
        tamanhos = c_f.sum(axis=1)
        linhas.append({
            'folha': int(f),
            'n_teste': int(mask.sum()),
            'cobertura': float(cobertos.mean()),
            'tam_medio': float(tamanhos.mean()),
            'pct_unitarios': float((tamanhos == 1).mean()),
            'pct_vazios': float((tamanhos == 0).mean()),
            'pct_duplos': float((tamanhos == 2).mean()),
        })
    return pd.DataFrame(linhas).sort_values('n_teste', ascending=False)

# imprime tabelas por desfecho
for alvo in ALVOS:
    print(f'\n=== {alvo} ===')
    df = cobertura_por_folha(
        y_test[alvo].values,
        resultados_locart[alvo]['conjuntos'],
        resultados_locart[alvo]['folhas_test'],
    )
    resultados_locart[alvo]['tabela_folhas'] = df
    print(df.to_string(index=False))


=== y_fisic ===
 folha  n_teste  cobertura  tam_medio  pct_unitarios  pct_vazios  pct_duplos
    34   134237   0.921043   1.349404       0.650596    0.000000    0.349404
    29   133572   0.880903   0.996227       0.996227    0.003773    0.000000
    13   123908   0.909126   1.140943       0.859057    0.000000    0.140943
     6   100526   0.912152   0.974693       0.974693    0.025307    0.000000
    26    75140   0.892880   0.903993       0.903993    0.096007    0.000000
    28    51078   0.884862   0.894965       0.894965    0.105035    0.000000
    18    30308   0.893065   1.731160       0.268840    0.000000    0.731160
    37    27022   0.929835   1.574384       0.425616    0.000000    0.574384
    25    25652   0.888664   0.915484       0.915484    0.084516    0.000000
    35    19419   0.921005   1.604254       0.395746    0.000000    0.604254
    38    19022   0.935653   1.543949       0.456051    0.000000    0.543949
    46    12486   0.902451   1.790726       0.209274    0.0

## 8. Comparação com MAPIE

Lado a lado, vê como Locart se posiciona contra LAC e APS. A diferença que
importa é nas folhas onde MAPIE estava por baixo do nominal.

In [9]:
linhas = []
for alvo in ALVOS:
    lac = baseline['lac'][alvo]
    aps = baseline['aps'][alvo]
    loc = resultados_locart[alvo]
    for nome, res in [('LAC', lac), ('APS', aps), ('Locart', loc)]:
        linhas.append({
            'desfecho': alvo,
            'metodo': nome,
            'cobertura': res['cobertura'],
            'tam_medio': res['tamanho_medio'],
            'pct_vazios': res['pct_vazios'],
            'pct_unitarios': res['pct_unitarios'],
            'pct_duplos': res['pct_duplos'],
        })

tabela_comparativa = pd.DataFrame(linhas)
print(tabela_comparativa.to_string(index=False))

desfecho metodo  cobertura  tam_medio  pct_vazios  pct_unitarios  pct_duplos
 y_fisic    LAC   0.906377   1.124264    0.000000       0.875736    0.124264
 y_fisic    APS   0.906377   1.124264    0.000000       0.875736    0.124264
 y_fisic Locart   0.903443   1.187146    0.022329       0.768196    0.209475
 y_psico    LAC   0.906337   1.159075    0.000000       0.840925    0.159075
 y_psico    APS   0.906337   1.159075    0.000000       0.840925    0.159075
 y_psico Locart   0.902099   1.234895    0.032064       0.700977    0.266959
  y_sexu    LAC   0.894597   0.952250    0.047750       0.952250    0.000000
  y_sexu    APS   0.894597   0.952250    0.047750       0.952250    0.000000
  y_sexu Locart   0.892399   1.013476    0.061362       0.863800    0.074838


## 9. Salvando

Guardo os resultados para o notebook 04 fazer a análise visual e a comparação
mais detalhada.

In [10]:
resultados_finais = {
    'locart': resultados_locart,
    'tabela_comparativa': tabela_comparativa,
    'alpha': ALPHA,
}

with open(SAIDA / 'resultados_locart.pkl', 'wb') as f:
    pickle.dump(resultados_finais, f)

print(f'Salvo em {SAIDA / "resultados_locart.pkl"}')

Salvo em /content/drive/MyDrive/projeto_violencia_mulher/conformal_violence/resultados_locart.pkl
